# 第 8 课：ASR 语言模型综合项目与闭卷验收

这是基础路线的结业项目。你将用同一条可复现流水线完成：

```text
训练语料 → 1/2/3-gram ARPA → 开发集 PPL → N-best
         → 声学/LM 融合 → 开发集调 LM scale → 测试集 WER
         → OpenFst lattice 最短路径 → 错误分析与实验报告
```

Notebook 成功运行只证明参考流水线可复现；你还必须完成最后的代码题和闭卷题，才算掌握。


## 0. 验收标准

- 能解释 PPL、WER、oracle WER 各自回答什么问题；
- 只在开发集选择 N-gram 阶数与融合权重；
- 测试集只做一次最终报告，不用其答案调参；
- 能用 OpenFst 复现 Python 融合得到的 1-best；
- 两道代码题全部通过，闭卷概念题至少 10/12；
- 能口头解释一次失败实验，不能只报一个分数。


In [ ]:
from pathlib import Path
from math import exp, log
import json
import subprocess

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('请从 learn_asr 项目或 notebooks 目录启动 Jupyter')

ROOT = find_project_root()
PREV = ROOT / 'openfst_lab' / 'lesson05'
LAB = ROOT / 'openfst_lab' / 'lesson08'
LAB.mkdir(parents=True, exist_ok=True)
words_path = PREV / 'words.txt'
arpa_paths = {n: PREV / f'tiny.{n}gram.arpa' for n in [1, 2, 3]}
missing = [str(path) for path in [words_path, *arpa_paths.values()] if not path.exists()]
if missing:
    raise FileNotFoundError('请先完整运行第 5 课，缺少：' + ', '.join(missing))

def run_wsl(*args, check=True):
    result = subprocess.run(
        ['wsl', '-d', 'Ubuntu', '--', *map(str, args)],
        text=True, capture_output=True, check=False, encoding='utf-8', errors='replace',
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'命令失败：{args}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}')
    return result

def to_wsl_path(path):
    resolved = Path(path).resolve()
    drive = resolved.drive.rstrip(':').lower()
    relative = resolved.relative_to(resolved.anchor).as_posix()
    return f'/mnt/{drive}/{relative}'

def write_lf(path, text):
    Path(path).write_text(text, encoding='utf-8', newline='\n')

for command in ['fstcompile', 'fstshortestpath', 'fstprint', 'fstinfo']:
    assert run_wsl('which', command).stdout.strip(), command
print('实验目录：', LAB)


## 1. 统一的 ARPA 评分器

评分器严格执行高阶匹配失败后的 backoff。句子概率包含 `<s>` 到首词，以及末词到 `</s>`。


In [ ]:
def parse_arpa(path):
    entries = {}
    order = 0
    for raw in Path(path).read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if line.startswith('\\') and line.endswith('-grams:'):
            order = int(line[1:].split('-')[0])
            entries[order] = {}
        elif order and line and not line.startswith('\\'):
            fields = line.split()
            probability = float(fields[0])
            ngram = tuple(fields[1:1 + order])
            backoff = float(fields[1 + order]) if len(fields) > 1 + order else 0.0
            entries[order][ngram] = (probability, backoff)
    return entries

ARPA_MODELS = {order: parse_arpa(path) for order, path in arpa_paths.items()}

def arpa_word_log10(model, word, history):
    max_order = max(model)
    accumulated_backoff = 0.0
    for history_length in range(min(len(history), max_order - 1), -1, -1):
        current_history = tuple(history[-history_length:]) if history_length else ()
        candidate = current_history + (word,)
        if candidate in model[len(candidate)]:
            return accumulated_backoff + model[len(candidate)][candidate][0]
        if current_history:
            accumulated_backoff += model[len(current_history)].get(current_history, (0.0, 0.0))[1]
    raise KeyError(word)

def sentence_lm_cost(model, text):
    tokens = ['<s>', *text.split(), '</s>']
    total_log10 = sum(arpa_word_log10(model, tokens[i], tokens[:i]) for i in range(1, len(tokens)))
    return -total_log10 * log(10)

def corpus_perplexity(model, sentences):
    total_cost = sum(sentence_lm_cost(model, sentence) for sentence in sentences)
    predicted_tokens = sum(len(sentence.split()) + 1 for sentence in sentences)  # 包含 </s>
    return exp(total_cost / predicted_tokens)

probe = 'jintian tianqi henhao'
for order, model in ARPA_MODELS.items():
    print(f'{order}-gram cost({probe}) = {sentence_lm_cost(model, probe):.6f}')


## 2. 先用开发文本比较困惑度

PPL 只评价文本概率预测，不直接等于 ASR WER。这里先选择开发文本 PPL 最低的阶数，再在开发 N-best 上调融合权重。测试参考答案不会参与这两个选择。


In [ ]:
DEV_TEXT = [
    'jintian tianqi henhao',
    'wo xihuan yuyin shibie',
]
ppl_by_order = {order: corpus_perplexity(model, DEV_TEXT) for order, model in ARPA_MODELS.items()}
for order, ppl in ppl_by_order.items():
    print(f'{order}-gram dev PPL = {ppl:.4f}')
selected_order = min(ppl_by_order, key=ppl_by_order.get)
selected_model = ARPA_MODELS[selected_order]
print('按开发集选择：', selected_order, '-gram')
assert selected_order in {1, 2, 3}


## 3. 开发集与测试集 N-best

每个 utterance 都有参考文本和一遍声学候选。`acoustic` 是负对数代价，越低越符合音频。这里的数据是可控教学样例，目的是验证调参协议，而不是宣称真实模型精度。


In [ ]:
DEV_UTTERANCES = [
    {
        'id': 'dev-1', 'reference': 'jintian tianqi henhao',
        'candidates': [
            ('jintian tianqi henhao', 2.6), ('jintian tianqi bucuo', 3.0),
            ('jintian xinqing henhao', 2.5), ('mingtian tianqi henhao', 2.2),
        ],
    },
    {
        'id': 'dev-2', 'reference': 'wo xihuan yuyin shibie',
        'candidates': [
            ('wo xihuan yuyin shibie', 2.8), ('wo xuexi yuyin shibie', 3.0),
            ('ni xihuan yuyin shibie', 2.4), ('wo xihuan yuyan moxing', 2.3),
        ],
    },
]

TEST_UTTERANCES = [
    {
        'id': 'test-1', 'reference': 'jintian xinqing bucuo',
        'candidates': [
            ('jintian xinqing bucuo', 2.5), ('mingtian xinqing henhao', 2.45),
            ('zuotian xinqing bucuo', 2.4), ('jintian tianqi bucuo', 3.2),
        ],
    },
    {
        'id': 'test-2', 'reference': 'ni xuexi yuyan moxing',
        'candidates': [
            ('ni xuexi yuyan moxing', 2.4), ('ni xihuan yuyan moxing', 2.7),
            ('wo xuexi ziran yuyan', 2.2), ('ni xuexi yuyin shibie', 2.45),
        ],
    },
]

def attach_lm_costs(utterances, model):
    return [
        {**utterance, 'candidates': [
            {'text': text, 'acoustic': acoustic, 'lm': sentence_lm_cost(model, text)}
            for text, acoustic in utterance['candidates']
        ]}
        for utterance in utterances
    ]

DEV_SCORED = attach_lm_costs(DEV_UTTERANCES, selected_model)
TEST_SCORED = attach_lm_costs(TEST_UTTERANCES, selected_model)
for utterance in DEV_SCORED:
    print(utterance['id'], 'reference=', utterance['reference'])
    for candidate in utterance['candidates']:
        print(f'  {candidate["text"]:30s} A={candidate["acoustic"]:.3f} LM={candidate["lm"]:.3f}')


## 4. WER 与融合解码

数据集 WER 必须累加所有 utterance 的编辑错误数与参考词数，再相除；不能简单平均每句话的 WER。


In [ ]:
def edit_distance(reference, hypothesis):
    rows, cols = len(reference) + 1, len(hypothesis) + 1
    table = [[0] * cols for _ in range(rows)]
    for i in range(rows): table[i][0] = i
    for j in range(cols): table[0][j] = j
    for i in range(1, rows):
        for j in range(1, cols):
            table[i][j] = min(
                table[i-1][j] + 1, table[i][j-1] + 1,
                table[i-1][j-1] + (reference[i-1] != hypothesis[j-1]),
            )
    return table[-1][-1]

def fuse_cost(acoustic_cost, lm_cost, lm_scale):
    return acoustic_cost + lm_scale * lm_cost

def decode_utterance(utterance, lm_scale):
    ranked = sorted(
        ({**candidate, 'total': fuse_cost(candidate['acoustic'], candidate['lm'], lm_scale)}
         for candidate in utterance['candidates']),
        key=lambda candidate: candidate['total'],
    )
    return ranked[0], ranked

def evaluate(utterances, lm_scale):
    errors = 0
    reference_words = 0
    rows = []
    for utterance in utterances:
        best, ranked = decode_utterance(utterance, lm_scale)
        ref = utterance['reference'].split()
        hyp = best['text'].split()
        utterance_errors = edit_distance(ref, hyp)
        errors += utterance_errors
        reference_words += len(ref)
        oracle_errors = min(edit_distance(ref, candidate['text'].split()) for candidate in ranked)
        rows.append({'id': utterance['id'], 'reference': utterance['reference'],
                     'hypothesis': best['text'], 'errors': utterance_errors,
                     'oracle_errors': oracle_errors})
    return {'wer': errors / reference_words, 'errors': errors, 'words': reference_words, 'rows': rows}

for scale in [0.0, 0.5, 1.0]:
    result = evaluate(DEV_SCORED, scale)
    print(f'LM scale={scale:.1f} dev WER={result["wer"]:.3f}')


## 5. 只在开发集调 LM scale

网格中可能有多个 scale 得到相同最低 WER。这里用较小 scale 作为确定性 tie-break，减少语言先验压过声学证据的风险。实际系统还应比较置信度、实体错误和跨域稳定性。


In [ ]:
scale_grid = [round(index * 0.1, 1) for index in range(21)]
dev_curve = [(scale, evaluate(DEV_SCORED, scale)['wer']) for scale in scale_grid]
best_dev_wer = min(wer for _, wer in dev_curve)
best_scale = min(scale for scale, wer in dev_curve if wer == best_dev_wer)
print('开发集曲线：')
print(' '.join(f'{scale:.1f}:{wer:.3f}' for scale, wer in dev_curve))
print(f'选择 LM scale={best_scale:.1f}，dev WER={best_dev_wer:.3f}')
assert best_dev_wer == 0.0
assert best_scale > 0.0


## 6. 冻结参数后评测测试集

现在才读取测试评测结果。若结果不好，应回到开发方案重新设计并重新建立独立测试集，不能盯着下面两句的答案反复调 scale。


In [ ]:
acoustic_only = evaluate(TEST_SCORED, 0.0)
tuned_result = evaluate(TEST_SCORED, best_scale)
print(f'声学基线 test WER={acoustic_only["wer"]:.3f} ({acoustic_only["errors"]}/{acoustic_only["words"]})')
print(f'融合系统 test WER={tuned_result["wer"]:.3f} ({tuned_result["errors"]}/{tuned_result["words"]})')
for row in tuned_result['rows']:
    print(f'{row["id"]}: ref={row["reference"]} | hyp={row["hypothesis"]} | err={row["errors"]}')
assert tuned_result['wer'] <= acoustic_only['wer']
assert all(row['oracle_errors'] == 0 for row in tuned_result['rows'])


## 7. 分析失败，再用 OpenFst 独立复现 1-best

参考运行中，开发集选择的系统降低了测试 WER，但 `test-1` 仍然错误；它的 `oracle_errors=0`，说明正确候选还在 N-best 中，问题属于**重排失败**而不是候选召回失败。此时不能偷看测试答案继续调 `LM scale`，而应在新的开发数据上研究更好的 LM、上下文或二遍模型。

Python 排序方便调参，但最终还要证明图上的最短路径得到同一结果。下面把每个候选作为一条独立路径，路径终止权重存放融合代价，再调用真实 `fstshortestpath`。


In [ ]:
def build_candidate_fst(utterance, lm_scale, stem):
    lines = []
    next_state = 1
    for candidate in utterance['candidates']:
        state = 0
        for word in candidate['text'].split():
            destination = next_state
            next_state += 1
            lines.append(f'{state} {destination} {word} 0')
            state = destination
        total = fuse_cost(candidate['acoustic'], candidate['lm'], lm_scale)
        lines.append(f'{state} {total:.9f}')
    text_path = LAB / f'{stem}.txt'
    fst_path = LAB / f'{stem}.fst'
    write_lf(text_path, '\n'.join(lines) + '\n')
    run_wsl(
        'fstcompile', '--acceptor=true',
        f'--isymbols={to_wsl_path(words_path)}', f'--osymbols={to_wsl_path(words_path)}',
        '--keep_isymbols=true', '--keep_osymbols=true',
        to_wsl_path(text_path), to_wsl_path(fst_path),
    )
    return fst_path

def shortest_acceptor_text(fst_path, stem):
    best_path = LAB / f'{stem}.best.fst'
    run_wsl('fstshortestpath', to_wsl_path(fst_path), to_wsl_path(best_path))
    printed = run_wsl('fstprint', '--acceptor=true', to_wsl_path(best_path)).stdout
    info = run_wsl('fstinfo', to_wsl_path(best_path)).stdout
    initial = int(next(line.split()[-1] for line in info.splitlines() if line.strip().startswith('initial state')))
    arcs = {}
    for line in printed.splitlines():
        fields = line.split()
        if len(fields) >= 3:
            arcs[int(fields[0])] = (int(fields[1]), fields[2])
    words = []
    state = initial
    while state in arcs:
        state, label = arcs[state]
        if label != '<eps>': words.append(label)
    return ' '.join(words), printed

test_example = TEST_SCORED[0]
candidate_fst = build_candidate_fst(test_example, best_scale, 'test_1_lattice')
fst_hypothesis, fst_printed = shortest_acceptor_text(candidate_fst, 'test_1_lattice')
python_hypothesis = decode_utterance(test_example, best_scale)[0]['text']
print('Python 1-best：', python_hypothesis)
print('OpenFst 1-best：', fst_hypothesis)
print('\n最短路径：\n' + fst_printed)
assert fst_hypothesis == python_hypothesis


## 8. 自动生成可复现实验报告

报告记录数据划分、模型选择依据、调参范围和最终指标。真实项目还应记录代码版本、随机种子、语料许可证、延迟、内存和实体专项指标。


In [ ]:
report = {
    'experiment': 'ASR LM zero-to-capstone',
    'ngram_order_selected_on_dev_ppl': selected_order,
    'dev_ppl_by_order': ppl_by_order,
    'lm_scale_grid': scale_grid,
    'lm_scale_selected_on_dev_wer': best_scale,
    'dev_wer': best_dev_wer,
    'test_acoustic_only_wer': acoustic_only['wer'],
    'test_fused_wer': tuned_result['wer'],
    'test_rows': tuned_result['rows'],
    'warning': 'controlled teaching candidates; not a production ASR benchmark',
}
report_path = LAB / 'capstone_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(report_path)
print(json.dumps(report, ensure_ascii=False, indent=2))


## 9. 闭卷代码题

不要复制上面的函数。先把上文折叠或重新打开一个空白 Notebook，再独立实现。允许使用 Python 基础语法，不允许调用本课参考函数。

1. `student_fuse(acoustic, lm, alpha)` 返回 cost 域融合代价；
2. `student_wer(reference, hypothesis)` 返回单句 word error rate。


In [ ]:
def student_fuse(acoustic, lm, alpha):
    # TODO：删除 pass，独立实现
    pass

def student_wer(reference, hypothesis):
    # TODO：删除 pass，独立实现；输入是两个以空格分词的字符串
    pass

code_results = []
try:
    code_results.append(('融合公式', abs(student_fuse(2.0, 3.0, 0.5) - 3.5) < 1e-9))
except Exception:
    code_results.append(('融合公式', False))
try:
    cases = [
        abs(student_wer('a b c', 'a b c') - 0.0) < 1e-9,
        abs(student_wer('a b c', 'a x c') - 1/3) < 1e-9,
        abs(student_wer('a b', 'a b c') - 0.5) < 1e-9,
    ]
    code_results.append(('WER', all(cases)))
except Exception:
    code_results.append(('WER', False))
for name, passed in code_results:
    print(('✅' if passed else '❌'), name)
print(f'代码题：{sum(passed for _, passed in code_results)}/2')


## 10. 闭卷概念题

每题尽量只写一个关键词或短语。不要在第一次作答前向上翻。


In [ ]:
questions = [
    '1. N-gram 的马尔可夫近似限制了什么？',
    '2. 未见过事件概率为零时，需要哪类技术？',
    '3. PPL 越高还是越低通常越好？',
    '4. OpenFst tropical cost 越高还是越低越好？',
    '5. L 的输出标签必须匹配 G 的哪一侧？',
    '6. G 中用于 backoff 的消歧标签是什么？',
    '7. 传统 HMM 解码图常写成什么？',
    '8. CTC topology 主要处理哪两个现象？',
    '9. 什么结构会共享候选的公共路径？',
    '10. 正确候选被剪掉后，只重排能否恢复？',
    '11. 融合权重应在开发集还是测试集调？',
    '12. top-k 中理论最小 WER 叫什么？',
]
for question in questions:
    print(question)

answers = ['', '', '', '', '', '', '', '', '', '', '', '']
expected = [
    {'历史长度', '上下文长度', 'history'}, {'平滑', 'smoothing'}, {'低', '越低越好'},
    {'低', '越低越好'}, {'输入', '输入侧'}, {'#0'}, {'hclg', 'h∘c∘l∘g'},
    {'blank和重复', 'blank/repeat', '空白和重复'}, {'lattice'}, {'不能', '否', 'no'},
    {'开发集', 'dev', 'validation'}, {'oracle wer', 'oraclewer'},
]

def normalize_answer(value):
    return str(value).strip().lower().replace(' ', '')

concept_score = 0
for index, (answer, accepted) in enumerate(zip(answers, expected), start=1):
    correct = normalize_answer(answer) in {normalize_answer(item) for item in accepted}
    concept_score += int(correct)
    print(('✅' if correct else '❌'), f'第 {index} 题')
print(f'概念题：{concept_score}/12；通过线为 10/12。')


## 11. 最终验收与下一阶段

基础阶段通过必须同时满足：

- [ ] 第 1～7 课各自达到要求分数；
- [ ] 本课代码题 2/2；
- [ ] 本课概念题至少 10/12；
- [ ] 能不看资料画出 `HCLG` 和 `CTC-TLG`；
- [ ] 能解释为什么开发集调参、测试集只做最终报告；
- [ ] 能从一次失败样本判断是候选召回失败还是重排失败。

通过后进入 [第 9 课：前沿 ASR 语言模型系统设计实验室](语言模型零基础_09_前沿ASR语言模型系统设计实验室.ipynb)：检索式 contextual ASR、N-best/neural LM 重打分、音频条件纠错、SpeechLLM adapter、幻觉检测与安全回退。
